In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str=Field(description="Detail about the feedback")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [6]:
structured_model=model.with_structured_output(EvaluationSchema)

In [7]:
essay = """
India in the Age of AI

Artificial Intelligence (AI) is changing the world quickly, and India has a great opportunity to grow with this technology. AI can help improve many areas such as healthcare, agriculture, education, and business.

India has many talented engineers, IT professionals, and students. The country also has a strong technology industry and many startups working on AI solutions. The government has introduced programs like "AI for All" to encourage the use of AI for the benefit of society.

AI can help farmers by predicting weather conditions, suggesting the best time for planting crops, and controlling pests. In healthcare, AI can help doctors diagnose diseases faster and provide better treatment, especially in rural areas. In education, AI-powered tools can create personalized learning experiences for students.

However, there are also challenges. Many rural areas still lack good internet access and digital skills. AI may also replace some jobs, making it important for people to learn new skills. Another concern is data privacy, as AI systems use large amounts of personal information.

To use AI successfully, India must focus on education, skill development, ethical AI practices, and strong data protection laws. Cooperation between the government, industries, and educational institutions is also necessary.

In conclusion, AI can help India achieve economic growth and improve the quality of life for its people. With responsible use and proper planning, India can become a global leader in the age of AI.
"""

In [9]:
prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n{essay}"
structured_model.invoke(prompt).score

7

In [13]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [16]:
def evaluate_language(state: UPSCState):

    prompt = f"""
    Evaluate the language quality of the following essay.
    Provide detailed feedback and assign a score out of 10.

    Essay:
    {state["essay"]}
    """

    output = structured_model.invoke(prompt)

    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [17]:
def evaluate_analysis(state: UPSCState):

    prompt = f"""
    Evaluate the depth of analysis of the following essay.
    Provide detailed feedback and assign a score out of 10.

    Essay:
    {state["essay"]}
    """

    output = structured_model.invoke(prompt)

    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [18]:
def evaluate_thought(state: UPSCState):

    prompt = f"""
    Evaluate the clarity of thought of the following essay.
    Provide detailed feedback and assign a score out of 10.

    Essay:
    {state["essay"]}
    """

    output = structured_model.invoke(prompt)

    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [19]:
def final_evaluation(state: UPSCState):

    summary_prompt = f"""
    Based on the following feedbacks,
    create a summarized feedback.

    Language Feedback:
    {state["language_feedback"]}

    Analysis Feedback:
    {state["analysis_feedback"]}

    Clarity Feedback:
    {state["clarity_feedback"]}
    """

    overall_feedback = model.invoke(
        summary_prompt
    ).content

    avg_score = (
        sum(state["individual_scores"])
        / len(state["individual_scores"])
    )

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }

In [22]:
graph = StateGraph(UPSCState)

graph.add_node(
    "evaluate_language",
    evaluate_language
)

graph.add_node(
    "evaluate_analysis",
    evaluate_analysis
)

graph.add_node(
    "evaluate_thought",
    evaluate_thought
)

graph.add_node(
    "final_evaluation",
    final_evaluation
)


# Start Connections
graph.add_edge(
    START,
    "evaluate_language"
)

graph.add_edge(
    START,
    "evaluate_analysis"
)

graph.add_edge(
    START,
    "evaluate_thought"
)


# Merge into Final Evaluation
graph.add_edge(
    "evaluate_language",
    "final_evaluation"
)

graph.add_edge(
    "evaluate_analysis",
    "final_evaluation"
)

graph.add_edge(
    "evaluate_thought",
    "final_evaluation"
)


# End
graph.add_edge(
    "final_evaluation",
    END
)

# Compile Workflow
workflow = graph.compile()


# ==========================
# Run Workflow
# ==========================
result = workflow.invoke(
    {
        "essay": essay
    }
)


In [23]:
print("\nLANGUAGE FEEDBACK")
print(result["language_feedback"])

print("\nANALYSIS FEEDBACK")
print(result["analysis_feedback"])

print("\nCLARITY FEEDBACK")
print(result["clarity_feedback"])

print("\nOVERALL FEEDBACK")
print(result["overall_feedback"])

print("\nAVERAGE SCORE")
print(result["avg_score"])


LANGUAGE FEEDBACK
The essay provides a comprehensive overview of the opportunities and challenges of AI in India. It highlights the potential benefits of AI in various sectors such as healthcare, agriculture, and education. The essay also acknowledges the challenges, including the lack of digital skills and internet access in rural areas, job displacement, and data privacy concerns. The writer emphasizes the need for education, skill development, and ethical AI practices. The essay is well-structured and easy to follow. However, some sentences are repetitive, and the writer could have provided more specific examples and data to support their claims. The conclusion effectively summarizes the main points and reiterates the potential of AI in India. Overall, the essay demonstrates a good understanding of the topic and presents a clear and coherent argument.

ANALYSIS FEEDBACK
The essay provides a good overview of the opportunities and challenges of AI in India. It effectively highlights 